In [10]:
import os
os.getcwd()
import pandas as pd

# Load full corpus
df = pd.read_csv("/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_cleaned.csv")

df_amz = df.copy()

In [11]:
import re
import pandas as pd

def clean_page_text(text, patterns):
    """
    Remove all regex patterns from a given text.
    Patterns must be a list of regex strings.
    """
    if pd.isna(text):
        return text
    
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE | re.DOTALL)
    # collapse extra whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned



In [12]:
import re
import pandas as pd

def final_super_clean(text):
    if pd.isna(text):
        return text

    # 1. Remove the classic PDF replacement character
    text = text.replace("�", " ")

    # 2. Remove the explicit weird control chars seen in reports
    text = text.replace("\x08", " ")   # BACKSPACE
    text = text.replace("\x05", " ")   # ENQUIRY

    # 3. Remove invisible Unicode characters (safe)
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00AD]", " ", text)

    # 4. Remove ALL ASCII control characters (0x00–0x1F, 0x7F)
    #    This catches any remaining PDF junk
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # 5. DO NOT remove non-ASCII (we keep em-dash, accents etc.)

    # 6. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [13]:
df["page_text_final"] = df["page_text_final"].astype(str).apply(final_super_clean)



In [14]:
patterns = [

    # ======================================================
    # 0. Company + Report Title Headers (Google / Alphabet)
    # ======================================================

    # "Google Environmental Report 2023", "Google Environmental Report | 2020", etc.
    r"\bGoogle\s+(Environmental|Sustainability|ESG)\s+Report(?:\s*\|\s*|\s+)\d{4}\b",

    # "Alphabet 2023 Environmental, Social, and Governance (ESG) Report"
    r"\bAlphabet\s+\d{4}\s+(Environmental|Sustainability|ESG)(?:,?\s+Social,?\s+and\s+Governance|\s+Report)\b.*",

    # "Alphabet Inc. 2023 Sustainability Report"
    r"\bAlphabet\s+Inc\.?\s+\d{4}\s+(Environmental|Sustainability|ESG)\s+Report\b.*",

    # Optional leading page number:
    # "23 Google Environmental Report 2022 ..."
    r"^\s*\d+\s+Google\s+(Environmental|Sustainability|ESG)\s+Report.*\d{4}\b",

    # ======================================================
    # 1. Navigation Ribbons / Section Menus
    # e.g. "Overview Our Approach Environment Social Governance Data Appendix"
    # ======================================================

    # A generic nav line composed of key sections (only when it’s mostly just them)
    r"^(Overview|Highlights|About\s+this\s+report|Our\s+approach|Our\s+company|Strategy|Governance|Environment|Environmental|Social|People|Community|Communities|Data|Metrics|Appendix|Appendices)"
    r"(?:\s+(Overview|Highlights|About\s+this\s+report|Our\s+approach|Our\s+company|Strategy|Governance|Environment|Environmental|Social|People|Community|Communities|Data|Metrics|Appendix|Appendices))*\s*$",

    # Pipes / bullets as separators:
    # "Highlights | Our approach | Planet | People | Governance | Data | Appendix"
    r"^(Highlights|Overview|About\s+this\s+report|Our\s+approach|Planet|People|Prosperity|Governance|Data|Appendix)"
    r"(?:\s*[\|\·>\u2022]\s*(Highlights|Overview|About\s+this\s+report|Our\s+approach|Planet|People|Prosperity|Governance|Data|Appendix))+\s*$",

    # Google-flavored planet/people/prosperity nav:
    r"(Planet|People|Prosperity)\s+(Planet|People|Prosperity)(\s+(Planet|People|Prosperity))*",

    # ======================================================
    # 2. Page Headers / Footers with Report Name + Page No.
    # ======================================================

    # "Google Environmental Report 2023  |  45"
    r"\bGoogle\s+(Environmental|Sustainability|ESG)\s+Report\s+\d{4}\s*[|–\-•]\s*\d+\b",

    # "Alphabet 2023 Sustainability Report | 12"
    r"\bAlphabet\s+\d{4}\s+(Environmental|Sustainability|ESG)\s+Report\s*[|–\-•]\s*\d+\b",

    # ======================================================
    # 3. "About this report" / methodology / boilerplate footers
    # ======================================================

    r"^About\s+this\s+report\b.*",
    r"^Report\s+scope\s+and\s+boundary\b.*",
    r"^Reporting\s+frameworks\b.*",
    r"\bThis\s+report\s+uses\s+data\s+from\s+Google’s\s+global\s+operations\b.*",
    r"\bUnless\s+otherwise\s+noted,\s+all\s+data\s+in\s+this\s+report\s+are\s+for\s+the\s+calendar\s+year\b.*",

    # Forward-looking statements / cautionary language
    r"This\s+report\s+contains\s+forward-?looking\s+statements.*",
    r"Forward-?looking\s+statements\s+are\s+inherently\s+subject\s+to\s+risks\s+and\s+uncertainties.*",

    # "For more information, see our Investor Relations website", etc.
    r"For\s+more\s+information,\s+please\s+see\s+our\s+(Investor\s+Relations|sustainability|environmental)\s+website.*",

    # ======================================================
    # 4. Data tables / index ribbons
    # ======================================================

    # "Data summary | Performance data | ESG data | Index"
    r"^(Data\s+summary|Performance\s+data|ESG\s+data|ESG\s+index)(\s*[\|\·>\u2022]\s*(Data\s+summary|Performance\s+data|ESG\s+data|ESG\s+index))+\s*$",

    # "GRI Content Index", "SASB Index", "TCFD Index"
    r"^(GRI|SASB|TCFD)\s+(Content\s+)?Index\b.*",

    # "Assurance statement" headers:
    r"^(Independent\s+)?Assurance\s+Statement\b.*",

    # ======================================================
    # 5. Multi-line headers combining year & report type
    # ======================================================

    # "2023 Environmental Report  |  Google"
    r"^20\d{2}\s+(Environmental|Sustainability|ESG)\s+Report\s*[|–\-•]\s+Google\b.*",

    # "Google | Environmental Report 2022"
    r"^Google\s*[|–\-•]\s+(Environmental|Sustainability|ESG)\s+Report\s+\d{4}\b.*",

    # ======================================================
    # 6. Legal / disclaimer / trademark boilerplate at page ends
    # ======================================================

    r"Google\s+LLC\s+is\s+not\s+obligated\s+to\s+update\s+these\s+forward-?looking\s+statements\.$",
    r"©\s*20\d{2}\s+Google\s+LLC\.?\s+All\s+rights\s+reserved\.$",
    r"Google\s+and\s+the\s+Google\s+logo\s+are\s+trademarks\s+of\s+Google\s+LLC\.$",

    # ======================================================
    # 7. Leading page numbers (for any Google page)
    # " 68 Climate Change ..." -> "Climate Change ..."
    # ======================================================

    r"^\s*\d+\s+(?=\S)",
]


In [15]:
df_amz["page_text_final_clean"] = df_amz["page_text_final"].apply(
    lambda x: clean_page_text(x, patterns)
)


In [16]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 600)
pd.set_option('display.max_colwidth', None)
df_amz[["page", "page_text_final", "page_text_final_clean"]]

page  \
0       0   
1       1   
2       2   
3       3   
4       4   
..    ...   
666    48   
667    49   
668    50   
669    51   
670    52   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [17]:
df_amz.shape
df_amz = df_amz.dropna(
    subset=["page_text_final", "page_text_final_clean"]
).copy()
df_amz = df_amz[df_amz["page_text_final_clean"].notna()].copy()

df_amz.shape
df_amz["page_text_final"] = df_amz["page_text_final_clean"]
df_amz = df_amz.drop(columns=["page_text_final_clean"])
df_amz = df_amz.rename(columns={"page_text_final": "page_text"})
df_amz.to_csv(
    "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_clean_final.csv",
    index=False
)

In [20]:
import csv
import re

# Path to your CleanFinal CSV
path = "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_clean_final.csv"

bad_rows = []

with open(path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)

    for i, row in enumerate(reader):
        # Expect exactly 4 columns: report, year, page, page_text
        if len(row) != 4:
            bad_rows.append((i, "❌ Wrong number of columns", row))
            continue

        text = row[3]

        # Check if page_text STARTS and ENDS with a quote
        starts_with_quote = text.startswith('"')
        ends_with_quote = text.endswith('"')

        if not (starts_with_quote and ends_with_quote):
            bad_rows.append((i, "❌ Missing surrounding quotes", row))


print("Total bad rows:", len(bad_rows))

# Show first few problematic rows
for r in bad_rows[:10]:
    print("\nRow", r[0], "-", r[1])
    print(r[2])



Total bad rows: 604

Row 0 - ❌ Missing surrounding quotes
['report_name', 'year', 'page', 'page_text']

Row 1 - ❌ Missing surrounding quotes
['report_2016.pdf', '2016', '1', 'Environmental sustainability at Google Our approach 02 Mission and values 03 About Google 07 About this report 07 Performance highlights and targets 08 Designing efficient data centers 12 Building an energy-efficient computing network 14 Free cooling: An efficient way to beat the heat 19 The zero waste moonshot 22 Driving our industry forward 23 Advancing renewable energy 25 Renewable energy is good for business 27 Reaching 100% renewable energy 29 New models for greening energy consumption 31 Accelerating the transition to clean energy 36 Our carbon footprint 39 Our journey to net zero carbon 41 Creating sustainable workplaces 42 Better buildings 44 The ecosystem in our backyard 46 A greener way to get there 47 Renewable energy starts at home 48 Conserving water in California and around the world 49 Designing out

In [22]:
import csv

input_path = "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_clean_final.csv"
output_path = "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_clean_final_fixed.csv"
with open(input_path, "r", encoding="utf-8") as infile, \
     open(output_path, "w", newline="", encoding="utf-8") as outfile:

    reader = csv.reader(infile)
    
    # QUOTING MINIMAL means we control who gets quoted manually
    writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)

    for row in reader:
        if len(row) != 4:
            # If structure unexpected, write as-is
            writer.writerow(row)
            continue

        report, year, page, text = row

        # FORCE quotes around page_text ONLY
        # Escape internal " → ""
        safe_text = text.replace('"', '""')
        safe_text = f'"{safe_text}"'

        # Write the fixed row
        writer.writerow([report, year, page, safe_text])

print("✔ Done! Saved to:", output_path)


✔ Done! Saved to: /Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/google_layout_all/corpus_pages_clean_final_fixed.csv
